# 00 — Run All

End-to-end notebook. Builds the LanceDB store (ingest + embeddings), runs both batch analyses (sentiment + topics), plots the results, and exposes an ad-hoc Q&A helper.

Run top-to-bottom. Every stage is resumable:
- **Ingest** skips files already seen (via `state/seen_files.json`).
- **Batches** skip entries already processed for the current `LLM_MODEL`.

Expected runtime on a cold run with `gemma3:4b`: ~25 min per batch. Re-runs only touch new entries, so they're fast.

## 1. Setup & ingest

Creates LanceDB tables and ingests any new `.html` entry files from `ENTRIES_DIR`. Embeddings come from `nomic-embed-text` via Ollama.

In [4]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from journal.config import ENTRIES_DIR, LANCE_ROOT, STATE_DIR
from journal.embed import OllamaEmbedder
from journal.ingest import Ingestor
from journal.store import Store

LANCE_ROOT.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)

store = Store(LANCE_ROOT)
store.create_tables()

ingestor = Ingestor(
    store=store,
    embedder=OllamaEmbedder(),
    state_path=STATE_DIR / "seen_files.json",
)
report = ingestor.scan_and_ingest(ENTRIES_DIR)
print(report)

df = store.entries_to_pandas()
print(f"total entries: {len(df):,}")
print(f"date range: {df['date'].min()} → {df['date'].max()}")

IngestReport(files_processed=0, entries_added=0, files_skipped=1831, errors=[])
total entries: 3,035
date range: 2020-06-30 → 2026-06-22


## 2. Run batch analyses (sentiment + topics)

Resumable: entries already analyzed for `LLM_MODEL` are skipped automatically, per question.

In [ ]:
import json

from journal.analyze import BatchAnalyzer
from journal.config import LLM_MODEL
from journal.llm import OllamaLLM
from journal.questions import default_registry

analyzer = BatchAnalyzer(
    store=store, llm=OllamaLLM(), registry=default_registry(),
    model=LLM_MODEL, max_workers=6,
)
result = analyzer.run_many(["sentiment", "topics"])
for report in result.reports:
    print(report)

## 3. Load analyses into DataFrames

In [ ]:
a = store.analyses_to_pandas()

sdf = (
    a[(a["question_id"] == "sentiment") & a["parsed_ok"]]
    .assign(
        level=lambda d: d["result_json"].apply(lambda x: json.loads(x)["level"]),
        score=lambda d: d["result_json"].apply(lambda x: json.loads(x)["score"]),
    )
    .merge(df[["id", "date", "day_of_week", "time_of_day"]],
           left_on="entry_id", right_on="id")
)

topics_df = (
    a[(a["question_id"] == "topics") & a["parsed_ok"]]
    .assign(topics=lambda d: d["result_json"].apply(lambda x: json.loads(x)["topics"]))
    .merge(df[["id", "date"]], left_on="entry_id", right_on="id")
)
topics_df["date"] = pd.to_datetime(topics_df["date"])
topics_exploded = topics_df.explode("topics")

## 4. Sentiment plots

Distribution by day of week and time of day, plus weekly average score.

In [ ]:
from journal.plots import diverging_sentiment_bar, score_over_time

day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
tod_order = ["morning", "afternoon", "evening", "night"]

diverging_sentiment_bar(sdf, "day_of_week", day_order, "Sentiment distribution by day of week").show()
diverging_sentiment_bar(sdf, "time_of_day", tod_order, "Sentiment distribution by time of day").show()
score_over_time(sdf, title="Average sentiment score over time (weekly)").show()

## 5. Topics plots

Top topics overall, then the top-8 topics tracked monthly.

In [ ]:
from journal.plots import top_n_bar, top_n_over_time

top_n_bar(topics_exploded["topics"], n=30, title="Top 30 topics across all entries").show()
top_n_over_time(
    topics_exploded, item_col="topics", date_col="date", n=8, freq="M",
    title="Top topics over time (monthly)",
).show()

## 6. Ad-hoc Q&A

Vector-retrieves the `k` most relevant entries for a free-form question, then asks the LLM to answer with citations. Edit the questions or add your own.

In [ ]:
from IPython.display import Markdown, display
from journal.query import answer_question

def ask(question, k=10):
    result = answer_question(
        store=store,
        embedder=OllamaEmbedder(),
        llm=OllamaLLM(),
        question=question,
        k=k,
    )
    display(Markdown(f"### Q: {question}\n\n{result.answer}"))
    display(Markdown(f"**Sources ({len(result.sources)}):**"))
    for i, row in result.sources.iterrows():
        snippet = row["text"][:200].replace("\n", " ")
        display(Markdown(f"- [{i}] {row['date']} — {snippet}..."))

In [ ]:
ask("When did I feel most grateful, and what about?")

In [ ]:
ask("What were the recurring sources of stress in 2021?")

In [ ]:
ask("How did my relationship with my parents come up over the years?")